## Perfiles Cliente-Producto: Preferencias de Atributos

### Por: Grupo 12 - ITBA
### Fecha: 2026-04-01

### Descripcion:
Creacion de features a nivel cliente basadas en preferencias de atributos de productos comprados: color dominante, diversidad, especializacion, etc.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

## Cargar datos

In [2]:
productos = pd.read_parquet('../../data/04_feature/productos_enriquecidos_regex.parquet')
ventas = pd.read_parquet('../../data/03_primary/ventas_limpias.parquet')
rfm = pd.read_parquet('../../data/04_feature/rfm_clientes.parquet')

df = ventas.merge(productos, on='Description', how='left')
print(f'Registros: {len(df):,}')

Registros: 397,884


## 1. Color Dominante por Cliente

In [3]:
COLORS = ['red', 'pink', 'white', 'blue', 'ivory', 'cream', 'green', 'black', 'silver', 'yellow']
color_flags = [f'has_{c}' for c in COLORS]

color_purchases = df.groupby('CustomerID')[color_flags].sum()
color_purchases['dominant_color'] = color_purchases[color_flags].idxmax(axis=1).str.replace('has_', '')
color_purchases['n_color_purchases'] = color_purchases[color_flags].sum(axis=1)
color_purchases['pct_with_color'] = (color_purchases['n_color_purchases'] / df.groupby('CustomerID').size()) * 100

perfiles = color_purchases[['dominant_color', 'pct_with_color']].copy()

## 2. Diversidad de Color (Shannon Entropy)

In [4]:
def shannon_entropy(row):
    counts = row[row > 0]
    if len(counts) == 0:
        return 0
    probs = counts / counts.sum()
    return entropy(probs, base=2)

perfiles['color_diversity'] = color_purchases[color_flags].apply(shannon_entropy, axis=1)
perfiles['is_color_specialist'] = perfiles['color_diversity'] <= 1.0

## 3. Material Dominante

In [5]:
MATERIALS = ['tin', 'wood', 'metal', 'paper', 'wooden', 'glass', 'ceramic']
material_flags = [f'has_{m}' for m in MATERIALS]

material_purchases = df.groupby('CustomerID')[material_flags].sum()
perfiles['dominant_material'] = material_purchases[material_flags].idxmax(axis=1).str.replace('has_', '')
perfiles['pct_with_material'] = (material_purchases[material_flags].sum(axis=1) / df.groupby('CustomerID').size()) * 100

## 4. Preferencia por Sets

In [6]:
set_data = df[df['is_set']].groupby('CustomerID').agg({
    'quantity_in_set': 'mean',
    'is_set': 'count'
}).rename(columns={'is_set': 'n_sets_purchased', 'quantity_in_set': 'avg_quantity_in_set'})

total_purchases = df.groupby('CustomerID').size()
set_data['pct_purchases_sets'] = (set_data['n_sets_purchased'] / total_purchases) * 100

perfiles = perfiles.merge(set_data[['avg_quantity_in_set', 'pct_purchases_sets']], left_index=True, right_index=True, how='left')
perfiles['avg_quantity_in_set'] = perfiles['avg_quantity_in_set'].fillna(0)
perfiles['pct_purchases_sets'] = perfiles['pct_purchases_sets'].fillna(0)

## 5. Resumen de Perfiles

In [7]:
perfiles.reset_index(inplace=True)
print(f'Clientes con perfiles: {len(perfiles):,}')
print(f'\nColumnas creadas: {list(perfiles.columns)}')
perfiles.describe()

Clientes con perfiles: 4,338

Columnas creadas: ['CustomerID', 'dominant_color', 'pct_with_color', 'color_diversity', 'is_color_specialist', 'dominant_material', 'pct_with_material', 'avg_quantity_in_set', 'pct_purchases_sets']


,CustomerID,pct_with_color,color_diversity,pct_with_material,avg_quantity_in_set,pct_purchases_sets
count,4338.000000,4338.000000,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,28.470526,1.741798,17.092737,10.970350,7.958288
std,1721.808492,17.409485,0.897047,14.511094,13.181746,9.428947
min,12346.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,13813.250000,18.181818,1.165808,8.139535,0.000000,0.000000
50%,15299.500000,27.005577,1.962685,14.598238,6.270979,5.989404
75%,16778.750000,36.363636,2.452657,22.304018,15.081250,11.111111
max,18287.000000,200.000000,3.165420,100.000000,72.000000,100.000000


## 6. Merge con RFM

In [8]:
rfm_enriched = rfm.merge(perfiles, on='CustomerID', how='left')
print(f'Shape: {rfm_enriched.shape}')
rfm_enriched.head()

Shape: (4338, 13)


,CustomerID,Recency,Frequency,Monetary,Cancel_rate,dominant_color,pct_with_color,color_diversity,is_color_specialist,dominant_material,pct_with_material,avg_quantity_in_set,pct_purchases_sets
0,12346,326,1,0.00,100.0,red,0.000000,0.000000,True,ceramic,100.000000,0.000000,0.000000
1,12347,2,7,4310.00,0.0,red,34.065934,2.577407,False,wooden,3.846154,38.307692,7.142857
2,12348,75,4,1797.24,0.0,pink,29.032258,2.113283,False,paper,6.451613,35.785714,45.161290
3,12349,19,1,1757.55,0.0,red,27.397260,1.970951,False,tin,21.917808,6.888889,12.328767
4,12350,310,1,334.40,0.0,red,29.411765,1.521928,False,metal,29.411765,0.000000,0.000000


## 7. Guardar tabla enriquecida

In [9]:
FEATURE_PATH = Path('../../data/04_feature/')
rfm_enriched.to_parquet(FEATURE_PATH / 'rfm_clientes_enriched.parquet', index=False)
print(f'Guardado: rfm_clientes_enriched.parquet ({rfm_enriched.shape[0]} x {rfm_enriched.shape[1]})')

Guardado: rfm_clientes_enriched.parquet (4338 x 13)
